# Nik Studio - video model test

One picture, one clip, about three seconds. Nothing here touches your
project.

This is a **test, not the tool**. It answers one question before any code
gets written: can a free Colab GPU animate your character well enough to
be worth building on?

The model is **LTX-Video (2B)**, chosen because it is small enough for a
free T4 and its licence allows commercial use. An earlier attempt used
CogVideoX-5B, which crashed the session - not for want of GPU, but
because free Colab only has 12.7GB of ordinary RAM and that model needs
more.

**Runtime > Change runtime type > T4 GPU** first, then run the cells in
order. Cell 3 takes 5-10 minutes, most of it downloading.


In [ ]:
# ======================================================================
# CELL 1 - packages, and check we really have a GPU
# ======================================================================
#
# Runtime > Change runtime type > T4 GPU  must be set BEFORE running this.
#
# If Colab offers "RESTART SESSION" after this cell, take it. The later
# cells are written to survive a restart.

!pip install -q "diffusers>=0.32" "transformers>=4.44" accelerate safetensors sentencepiece bitsandbytes imageio-ffmpeg

import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime > Change runtime type > T4 GPU, then run again."
    )

major, minor = torch.cuda.get_device_capability()

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
print(f"Compute capability: {major}.{minor}")

# Do not ask torch.cuda.is_bf16_supported() - it answers True on a T4,
# because torch emulates bfloat16 in software rather than refusing, and
# the emulation is slower than it is worth. Ask the hardware: compute
# capability 8.0 (Ampere) or newer is where bfloat16 is real.
DTYPE = torch.bfloat16 if major >= 8 else torch.float16

print("Using", "bfloat16 (native)" if major >= 8
      else "float16 (no real bfloat16 on this card)")


In [ ]:
# ======================================================================
# CELL 2 - the picture to animate, and what should happen in it
# ======================================================================

from pathlib import Path

from PIL import Image

# LTX needs both sides to be a multiple of 32. 704x384 is close to 16:9
# and is deliberately small - this run is to find out whether the child
# moves, not to make the final clip. Bigger comes after that works.
WIDTH, HEIGHT = 704, 384

# --------------------------------------------------------------- picture

# Drive is checked first. Nik Studio moves finished images off Drive and
# onto your PC, so usually nothing is there and the upload box appears -
# pick Images\Scene01.png from your episode folder.

def find_in_drive():

    root = Path("/content/drive/MyDrive")

    if not root.exists():
        return None

    for pattern in ("NikStudio/**/Scene01.*", "NikStudio/**/Images/*.png"):
        for found in sorted(root.glob(pattern)):
            if found.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp"):
                return found

    return None


try:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

except ImportError:
    pass

SOURCE = find_in_drive()

if SOURCE:
    print("Found in Drive:", SOURCE)
else:
    print("Nothing in Drive - upload one image (Images\\Scene01.png).")

    from google.colab import files

    SOURCE = Path("/content") / next(iter(files.upload()))

# Cover and crop rather than squash - a stretched child is not a fair
# test of the model.
picture = Image.open(SOURCE).convert("RGB")

scale = max(WIDTH / picture.width, HEIGHT / picture.height)

picture = picture.resize(
    (round(picture.width * scale), round(picture.height * scale)),
    Image.LANCZOS,
)

left = (picture.width - WIDTH) // 2
top = (picture.height - HEIGHT) // 2

IMAGE = picture.crop((left, top, left + WIDTH, top + HEIGHT))

display(IMAGE)

# ---------------------------------------------------------------- prompt

# This describes the MOVEMENT, not the picture - the picture is already
# there. LTX wants a long, plain, physical description of what happens.
# One clear action beats three vague ones.

PROMPT = (
    "The little boy claps his hands together and bounces up and down "
    "with excitement, laughing. His head tilts and his arms swing. Soap "
    "bubbles drift slowly upward around him. The camera stays still. "
    "Pixar style 3D animation, smooth natural motion, bright and cheerful."
)

NEGATIVE = (
    "worst quality, blurry, jittery, distorted, deformed face, "
    "static image, no movement, extra limbs, watermark, text"
)

# 65 frames at 24fps is under three seconds. Small on purpose: decoding
# is where the memory goes, and a long clip is what kills the session.
FRAMES = 65
STEPS = 40
FPS = 24

print("\n", PROMPT)


In [ ]:
# ======================================================================
# CELL 3 - make the clip.  Roughly 5-10 minutes, most of it downloading
# ======================================================================
#
# Why this is written the way it is:
#
# The model itself is small (2B). The thing that crashed the last attempt
# is the T5 text encoder, which is 9GB on its own - more than the 12.7GB
# of ordinary RAM a free Colab has, once everything else is loaded too.
#
# So the text encoder is loaded in 8-bit and sent straight to the GPU,
# never passing through RAM at full size. That takes it from 9GB to about
# 4.7GB, and the whole thing then fits on the card with room to spare.

import gc

import torch

from diffusers import LTXImageToVideoPipeline

from transformers import BitsAndBytesConfig, T5EncoderModel

MODEL = "Lightricks/LTX-Video"

# Colab restarts the runtime after installing packages, which wipes cell
# 1. Work the precision out again rather than stop with a NameError.
if "DTYPE" not in globals():

    DTYPE = (
        torch.bfloat16
        if torch.cuda.get_device_capability()[0] >= 8
        else torch.float16
    )

# Anything left from an earlier attempt is still holding the card.
for leftover in ("pipe", "text_encoder"):
    if leftover in globals():
        del globals()[leftover]

gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------ the model

import time

started = time.time()

text_encoder = T5EncoderModel.from_pretrained(
    MODEL,
    subfolder="text_encoder",
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="auto",
)

pipe = LTXImageToVideoPipeline.from_pretrained(
    MODEL,
    text_encoder=text_encoder,
    torch_dtype=DTYPE,
)

# The text encoder is already on the GPU in 8-bit; this moves the rest.
pipe.transformer.to("cuda")
pipe.vae.to("cuda")

# Decoding 65 frames in one piece is what runs the card out of memory.
pipe.vae.enable_tiling()

print(f"Model ready in {time.time() - started:.0f}s.")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f}GB\n")

# --------------------------------------------------------- the generation

started = time.time()

frames = pipe(
    image=IMAGE,
    prompt=PROMPT,
    negative_prompt=NEGATIVE,
    width=WIDTH,
    height=HEIGHT,
    num_frames=FRAMES,
    frame_rate=FPS,
    num_inference_steps=STEPS,
    guidance_scale=3.0,
    generator=torch.Generator("cpu").manual_seed(42),
).frames[0]

print(f"Done in {(time.time() - started) / 60:.1f} minutes.")


In [ ]:
# ======================================================================
# CELL 4 - watch it, and keep a copy
# ======================================================================

from pathlib import Path

from diffusers.utils import export_to_video

OUT = Path("/content/drive/MyDrive/NikStudio/VideoTest")

if not OUT.parent.parent.exists():
    OUT = Path("/content")      # no Drive - keep it beside the notebook

OUT.mkdir(parents=True, exist_ok=True)

clip = OUT / "Scene01_test.mp4"

export_to_video(frames, str(clip), fps=globals().get("FPS", 24))

print("Saved:", clip)

from IPython.display import Video, display

display(Video(str(clip), embed=True, width=704))

# ----------------------------------------------------------------------
# One question: does the child MOVE, and is he still your character?
#
#   Yes                  -> the video backend gets built.
#   Face changed/melted  -> fewer steps or a shorter clip; fixable.
#   Barely moves         -> the prompt needs a stronger physical action.
#   Smeary or flickering -> float16 on a T4, or the clip is too long.
# ----------------------------------------------------------------------
